# 15. Permissions — Design Deep Agents authorization boundaries first

Permissions are part of the agent design, not a final hardening pass. This chapter shows how to reason about tool risk, targets, approval messages, and default-deny behavior.

**Learning goals**
- Build a simple permission matrix for agent tools.
- Evaluate allow/approve/deny decisions deterministically.
- Write approval messages that explain the risk clearly.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

In [ ]:
# Observability setup — disabled when keys are not present.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")
lf_config = {}

## 15.1 Permission matrix

A permission matrix makes policy visible before code executes. It should connect each tool to the targets it may read, write, modify, or never touch.


In [ ]:
permission_matrix = {
    "read_file": "allow",
    "write_file": "approve",
    "edit_file": "approve",
    "execute_shell": "sandbox-only",
    "delete_file": "deny",
}

permission_matrix

## 15.2 Policy Evaluator

A policy evaluator turns the matrix into a repeatable decision. Even a small deterministic evaluator is better than scattered permission checks.


In [ ]:
def decide(tool_name: str) -> str:
    return permission_matrix.get(tool_name, "deny")

for tool in ["read_file", "write_file", "delete_file", "unknown"]:
    print(tool, "=>", decide(tool))

## 15.3 Approval messages

Approval prompts should be specific enough for a human to make a real decision. They should name the tool, target, action, and reason for escalation.


In [ ]:
def approval_message(tool_name: str, target: str) -> str:
    policy = decide(tool_name)
    if policy != "approve":
        return f"{tool_name}: {policy}"
    return f"Approval required: {tool_name} will modify {target}"

approval_message("write_file", "docs/example.md")

---

## Summary

| Item | Content |
|---|---|
| **Covered** | permission matrices, approval messages, sandbox boundaries, and deny-by-default policies |
| **Core idea** | Start from a small deterministic contract before adding model calls or external services. |
| **Next step** | Follow the linked course notebooks and official reference notes listed in this chapter. |

## Reference docs

- [`permissions.md`](../../docs/deepagents/permissions.md)
- [`tools.md`](../../docs/deepagents/tools.md)
- [`sandboxes.md`](../../docs/deepagents/sandboxes.md)
